> 在线讲义：[第 32 章 · Triton 编程模型：一个 program 处理一块数据](https://wikieden.github.io/pytorch-inside-out/lessons/32-triton-model.html) · 本书《PyTorch 从里到外》

# 第 32 章 · Triton 编程模型：一个 program 处理一块数据

本章回答：Triton 写 kernel 和 CUDA 写 kernel 有什么不同？`program_id`、`tl.arange`、mask、`tl.load` / `tl.store` 各是什么？grid 怎么定？一行一个 program 的规约（比如 softmax）是怎么组织的？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/32-triton-model.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch14「使用 OpenAI Triton 编写自定义内核」（`ch14.md` 第 1109–1143 行）、「Triton 编程模型」（第 1144–1242 行，向量加法示例）、「在 Triton 中访问共享内存」（第 1243–1256 行）、「调优内核启动参数」与「自动调优」（第 1324–1387 行）；Triton 官方教程 01 vector add、02 fused softmax（〔在线〕triton-lang.org）；CUDA 讲义 第 24 章 Tile DSL

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 grid 与 mask | cdiv、offs < n | `programs(2500, 1024)` | 3 个 program，最后一个 452 个有效 |
| 第二步 分块算 | 一块一块 | `add_relu_blocked(x, y)` | 和 torch.relu(x + y) 一样 |
| 第三步 一行一个 program | 越界补 −∞ | `row_softmax_blocked(a, 1024)` | 和 torch.softmax 一样 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `cdiv`**（来自 `ch32_triton_model.py`）：下面要用到的小工具。

In [ ]:
def cdiv(a: int, b: int) -> int:
    return (a + b - 1) // b

**函数 `add_relu_blocked`**（来自 `ch32_triton_model.py`）：按 Triton 的方式一块一块算：offs = pid × BLOCK + arange，mask 挡越界。

In [ ]:
def add_relu_blocked(x: torch.Tensor, y: torch.Tensor, block: int = 1024) -> torch.Tensor:
    """按 Triton 的方式一块一块算：offs = pid × BLOCK + arange，mask 挡越界。"""
    n = x.numel()
    out = torch.empty_like(x)
    for pid in range(cdiv(n, block)):
        offs = pid * block + torch.arange(block)
        mask = offs < n
        o = offs[mask]
        out[o] = torch.clamp(x[o] + y[o], min=0)
    return out

**函数 `programs`**（来自 `ch32_triton_model.py`）：每个 program 的 (pid, offs 起点, offs 终点, mask 为真的个数)。

In [ ]:
def programs(n: int, block: int) -> list:
    """每个 program 的 (pid, offs 起点, offs 终点, mask 为真的个数)。"""
    return [(pid, pid * block, pid * block + block - 1, min(block, n - pid * block)) for pid in range(cdiv(n, block))]

**函数 `row_softmax_blocked`**（来自 `ch32_triton_model.py`）：一行一个 program；越界位置补 −∞，exp 后正好是 0。

In [ ]:
def row_softmax_blocked(x: torch.Tensor, block: int) -> torch.Tensor:
    """一行一个 program；越界位置补 −∞，exp 后正好是 0。"""
    rows, N = x.shape
    out = torch.empty_like(x)
    cols = torch.arange(block)
    for pid in range(rows):
        row = torch.full((block,), float("-inf"))
        row[cols < N] = x[pid]
        e = torch.exp(row - row.max())
        out[pid] = (e / e.sum())[:N]
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 n = 2500、BLOCK = 1024：", programs(2500, 1024))
x, y = torch.randn(2500), torch.randn(2500)
print("分块算的 add_relu 与 torch 一致：", torch.equal(add_relu_blocked(x, y), torch.relu(x + y)))
a = torch.randn(4, 1000)

In [ ]:
print("\n第三步 一行一个 program 的 softmax（BLOCK = 1024）与 torch 一致：",
      torch.allclose(row_softmax_blocked(a, 1024), torch.softmax(a, -1), atol=1e-6))

## 3. 代码里才看得到的

1. **模拟版是一个 program 接一个跑**：真 Triton 是所有 program 同时跑，结果一样、速度不同。
2. **越界补 −∞ 才对**：把 `row_softmax_blocked` 的 −∞ 改成 0，结果就错了。

## 4. 自己改一改

- 把补的值改成 0，看 softmax 结果哪里不对。
- `programs(1000, 256)` 先自己算再跑。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> CUDA 写一个线程，Triton 写一个 program 管一块；
> program_id 定位，arange 拿下标，mask 挡越界，load、store 整块读写；
> grid 向上取整；行规约一行一个 program，越界补不影响结果的值；块大小交给 autotune。